In [1]:
# Cell 1 — imports + config
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Config:
    dim = 32
    num_experts = 4
    hidden = 64
    batch_size = 256
    steps = 1500
    lr = 1e-3
    lb_weight = 0.1
    aux_weight = 0.5
    top_k = 1

cfg = Config()
print(cfg.__dict__)

{}


In [2]:
# Cell 2 — clustered Gaussian mixture dataset

torch.manual_seed(0)

cluster_means = torch.tensor([
    +3.0,   # Expert 0 cluster center
    -3.0,   # Expert 1 cluster center
    +6.0,   # Expert 2 cluster center
    -6.0    # Expert 3 cluster center
]).view(cfg.num_experts, 1)

def make_batch(batch_size):
    expert_ids = torch.randint(0, cfg.num_experts, (batch_size,))
    x = torch.randn(batch_size, cfg.dim)

    for i in range(batch_size):
        eid = int(expert_ids[i])
        x[i] += cluster_means[eid]

    return x.to(device), expert_ids.to(device)

In [3]:
# Cell 3 — expert-specific transforms (ground truth)

W0 = torch.randn(cfg.dim, cfg.dim)
W1 = torch.randn(cfg.dim, cfg.dim)
W2 = torch.randn(cfg.dim, cfg.dim)
W3 = torch.randn(cfg.dim, cfg.dim)

W0 = W0.to(device)
W1 = W1.to(device)
W2 = W2.to(device)
W3 = W3.to(device)

def expert_map(x, eid):
    if eid == 0:
        return x @ W0
    elif eid == 1:
        return torch.relu(x @ W1)
    elif eid == 2:
        return torch.sin(x @ W2)
    elif eid == 3:
        return torch.tanh(x @ W3)

In [4]:
# Cell 4 — shared Expert module

class Expert(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, dim)
        )

    def forward(self, x):
        return self.net(x)

In [6]:
# Cell 6 — Mask-MoE with manifold (identity + repulsion + supervised routing)

class MaskMoE_Manifold(nn.Module):
    def __init__(self, dim, num_experts, hidden, id_dim=16):
        super().__init__()
        self.num_experts = num_experts
        self.id_dim = id_dim

        self.router = nn.Sequential(
            nn.Linear(dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_experts)
        )

        self.identities = nn.Parameter(torch.randn(num_experts, id_dim))

        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim + id_dim, hidden),
                nn.ReLU(),
                nn.Linear(hidden, dim)
            )
            for _ in range(num_experts)
        ])

    def forward(self, x):
        B = x.size(0)

        logits = self.router(x)
        probs = F.softmax(logits, dim=-1)

        top1 = torch.argmax(probs, dim=-1)
        one_hot = F.one_hot(top1, num_classes=self.num_experts).float()

        outputs = torch.zeros_like(x)
        for e in range(self.num_experts):
            mask = one_hot[:, e].bool()
            if mask.any():
                id_vec = self.identities[e].expand(mask.sum(), self.id_dim)
                x_e = torch.cat([x[mask], id_vec], dim=-1)
                outputs[mask] = self.experts[e](x_e)

        frac_tokens = one_hot.mean(dim=0)
        mean_gate = probs.mean(dim=0)
        lb_loss = self.num_experts * torch.sum(frac_tokens * mean_gate)

        norm_id = F.normalize(self.identities, dim=-1)
        sim = norm_id @ norm_id.T
        repulsion_loss = (sim - torch.eye(self.num_experts, device=x.device)).pow(2).sum()

        stats = {
            "lb_loss": lb_loss,
            "entropy": (-probs * probs.clamp(min=1e-8).log()).sum(dim=-1).mean().item(),
            "ia": probs.max(dim=-1).values.mean().item(),
            "repulsion": repulsion_loss.item()
        }

        return outputs, stats, logits, repulsion_loss

In [7]:
# Cell 7 — mse loss + expert usage helper

def mse_loss(pred, target):
    return ((pred - target) ** 2).mean()

def get_expert_usage(probs, num_experts):
    top1 = torch.argmax(probs, dim=1)
    usage = torch.bincount(top1, minlength=num_experts)
    return usage


In [9]:
# Cell 9 — training Mask-MoE with manifold on same Gaussian task

mask_moe = MaskMoE_Manifold(cfg.dim, cfg.num_experts, cfg.hidden).to(device)
opt2 = torch.optim.Adam(mask_moe.parameters(), lr=cfg.lr)

for step in range(cfg.steps + 1):

    # 1. Make Gaussian batch
    x, expert_ids = make_batch(cfg.batch_size)

    # 2. Compute ground-truth expert_map output
    y = torch.stack([expert_map(x[i], int(expert_ids[i])) for i in range(cfg.batch_size)], dim=0)

    # 3. Forward pass through Mask-MoE manifold
    pred, stats, logits, repulsion_loss = mask_moe(x)

    # 4. Compute losses
    loss_main = mse_loss(pred, y)
    loss_lb = stats["lb_loss"]
    loss_route = F.cross_entropy(logits, expert_ids)

    # 5. Total loss = main + LB + routing + manifold repulsion
    loss = (
        loss_main +
        cfg.lb_weight * loss_lb +
        cfg.aux_weight * loss_route +
        0.1 * repulsion_loss
    )

    # 6. Backprop + update
    opt2.zero_grad()
    loss.backward()
    opt2.step()

    # 7. Print stats every 100 steps
    if step % 100 == 0:
        probs = F.softmax(logits, dim=-1)
        usage = get_expert_usage(probs, cfg.num_experts)
        print(
            f"[Mask-MoE] Step {step} | "
            f"Loss {loss_main.item():.4f} | "
            f"Ent {stats['entropy']:.4f} | "
            f"LB {loss_lb.item():.4f} | "
            f"IA {stats['ia']:.4f} | "
            f"Route {loss_route.item():.4f} | "
            f"Rep {stats['repulsion']:.4f}"
        )
        print("Mask-MoE expert usage:", usage)


[Mask-MoE] Step 0 | Loss 107.7345 | Ent 0.9650 | LB 2.2929 | IA 0.5963 | Route 2.0911 | Rep 0.6205
Mask-MoE expert usage: tensor([  0,  15, 241,   0], device='cuda:0')
[Mask-MoE] Step 100 | Loss 62.4182 | Ent 0.6714 | LB 1.1499 | IA 0.6248 | Route 0.5853 | Rep 0.5355
Mask-MoE expert usage: tensor([ 22,  18, 100, 116], device='cuda:0')
[Mask-MoE] Step 200 | Loss 20.6441 | Ent 0.6444 | LB 1.0407 | IA 0.6457 | Route 0.4654 | Rep 0.5121
Mask-MoE expert usage: tensor([48, 48, 84, 76], device='cuda:0')
[Mask-MoE] Step 300 | Loss 12.2616 | Ent 0.5752 | LB 1.0266 | IA 0.7167 | Route 0.3467 | Rep 0.4866
Mask-MoE expert usage: tensor([60, 54, 79, 63], device='cuda:0')
[Mask-MoE] Step 400 | Loss 9.2431 | Ent 0.4828 | LB 1.0102 | IA 0.7977 | Route 0.2328 | Rep 0.4561
Mask-MoE expert usage: tensor([54, 65, 72, 65], device='cuda:0')
[Mask-MoE] Step 500 | Loss 9.2244 | Ent 0.3785 | LB 1.0005 | IA 0.8645 | Route 0.1491 | Rep 0.4250
Mask-MoE expert usage: tensor([63, 73, 64, 56], device='cuda:0')
[Mask